In [31]:
from langchain_aws import ChatBedrockConverse
from langchain_ollama import ChatOllama
from typing import TypedDict, Literal,List,Dict,Annotated
from langgraph.graph import StateGraph,START,END,MessagesState
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage,BaseMessage,ToolMessage
import operator
from langgraph.prebuilt.tool_node import ToolNode
from langchain_core.tools import tool
import uuid
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command,interrupt



In [32]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=100)

In [33]:
class QuerytypeSchema(BaseModel):
    type:Literal['factual','opinion','how-to'] = Field(description="type of user query")

class StateSchema(TypedDict):
    query: str
    type: str
    answer: str
  
    

In [34]:
def classify_query(state:StateSchema):
    structured_response_llm = llm.with_structured_output(QuerytypeSchema)
    prompt = f"""find the type of the query - {state['query']}."""
    res = structured_response_llm.invoke(prompt)
    # print("====== classify query node ====")
    return {'type': res.type}

def explain_node(state:StateSchema):
    prompt = f"""generate an small explaination for the query - {state['query']}."""
    res = llm.invoke(prompt)
    # print("====== explain node ====")
    return {'answer': res.content}

def opinion_node(state:StateSchema):
    prompt = f"""generate an small opinion-style response for the query - {state['query']}."""
    res = llm.invoke(prompt)
    # print("====== opinion node ====")
    return {'answer': res.content}

def guide_node(state:StateSchema):
    prompt = f"""generate an small step by step guaidance for the query - {state['query']}."""
    res = ""
    for i in llm.stream(prompt):
        print(i.content,end='')
        res += i.content
    # print("====== guide node ====")
    return {'answer': res}

def check_condition(state:StateSchema):
    a = interrupt("Do you want to approve for check_condition")

    print("Approver command : ",a)
    
    if state['type'] == 'factual':
        return 'explain_node'
    elif state['type'] == 'opinion':
        return 'opinion_node'
    return 'guide_node'

    


In [35]:
graph = StateGraph(StateSchema)
graph.add_node('classify_query_node',classify_query)
graph.add_node('explain_node',explain_node)
graph.add_node('opinion_node',opinion_node)
graph.add_node('guide_node',guide_node)

graph.add_edge(START,'classify_query_node')
graph.add_conditional_edges('classify_query_node',check_condition,{'explain_node':'explain_node','opinion_node':'opinion_node','guide_node':'guide_node'})
graph.add_edge('explain_node',END)
graph.add_edge('opinion_node',END)
graph.add_edge('guide_node',END)

conn = sqlite3.connect("database/data.db",check_same_thread=False)
workflow = graph.compile(checkpointer=SqliteSaver(conn=conn))

In [36]:
print(workflow.get_graph().draw_ascii())

                            +-----------+                            
                            | __start__ |                            
                            +-----------+                            
                                  *                                  
                                  *                                  
                                  *                                  
                      +---------------------+                        
                      | classify_query_node |                        
                      +---------------------+                        
                   .....          .          .....                   
               ....               .               ....               
            ...                   .                   ...            
+--------------+           +------------+           +--------------+ 
| explain_node |           | guide_node |           | opinion_node | 
+--------------+*** 

In [37]:
thread_id = uuid.uuid4()
config = {'configurable':{'thread_id':thread_id}}
res = workflow.invoke({'query':'How to buid a langgraph'},config=config)
res

{'query': 'How to buid a langgraph',
 'type': 'how-to',
 '__interrupt__': [Interrupt(value='Do you want to approve for check_condition', id='58081b9818505f8667057caf4a2eb2de')]}

In [ ]:
workflow.invoke(Command(resume="You can execute now"),config=config)


Approver command :  You can execute now
Building a language graph involves several steps that can be broken down into manageable tasks. Here’s a simple guide to help you get started:

### Step 1: Define Your Objectives and Scope
- **Identify What You Want to Achieve**: Determine the purpose of your language graph, whether it's for research, analysis, or creating a knowledge base.
- **Define the Scope**: Decide on the domain or topic you want to focus on. This will help in narrowing down the types of relationships and entities you need.

### Step 2: Gather Data
- **Collect Information**: Collect relevant data from multiple sources such as books, articles, websites, databases, etc.
- **Organize Your Data**: Categorize your collected information into different topics or themes. This will help in structuring the graph effectively.

### Step 3: Choose a Graphing Tool
- **Select a Suitable Tool**: There are several tools available for building language graphs such as Neo4j, Microsoft Power B

{'query': 'How to buid a langgraph',
 'type': 'how-to',
 'answer': 'Building a language graph involves several steps that can be broken down into manageable tasks. Here’s a simple guide to help you get started:\n\n### Step 1: Define Your Objectives and Scope\n- **Identify What You Want to Achieve**: Determine the purpose of your language graph, whether it\'s for research, analysis, or creating a knowledge base.\n- **Define the Scope**: Decide on the domain or topic you want to focus on. This will help in narrowing down the types of relationships and entities you need.\n\n### Step 2: Gather Data\n- **Collect Information**: Collect relevant data from multiple sources such as books, articles, websites, databases, etc.\n- **Organize Your Data**: Categorize your collected information into different topics or themes. This will help in structuring the graph effectively.\n\n### Step 3: Choose a Graphing Tool\n- **Select a Suitable Tool**: There are several tools available for building language

In [39]:
chat_history = list(workflow.get_state_history(config=config))
for msg in chat_history[::-1]:
    print(msg)

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '816583ef-b9e2-4fe2-a7bc-336f1605bcb5', 'checkpoint_ns': '', 'checkpoint_id': '1f1b4cbd-da48-63a4-bfff-ab578355a7e2'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-09-20T08:18:31.510298+00:00', parent_config=None, tasks=(PregelTask(id='b1f6f988-f690-2be3-b0ea-85a92561b518', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'query': 'How to buid a langgraph'}),), interrupts=())
StateSnapshot(values={'query': 'How to buid a langgraph'}, next=('classify_query_node',), config={'configurable': {'thread_id': '816583ef-b9e2-4fe2-a7bc-336f1605bcb5', 'checkpoint_ns': '', 'checkpoint_id': '1f1b4cbd-da4b-62b8-8000-a767867dec94'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-20T08:18:31.511417+00:00', parent_config={'configurable': {'thread_id': '816583ef-b9e2-4fe2-a7bc-336f1605bcb5', 'checkpoint_ns': ''